# Data Exploration und Cleaning

Dieses Notebook liest die SMS-Spam-Rohdaten ein, prüft Datenqualität, Klassenverteilung, Encoding-Themen, HTML-Entities und Duplikate. Am Ende wird ein deduplizierter CSV-Datensatz nach `data/01_cleaned/sms_cleaned.csv` geschrieben.

In [ ]:
from pathlib import Path
import html
import re

import pandas as pd
from IPython.display import display

RAW_PATH = Path("data/00_raw/SMSSpamCollection")
CLEAN_DIR = Path("data/01_cleaned")
CLEAN_PATH = CLEAN_DIR / "sms_cleaned.csv"

pd.set_option("display.max_colwidth", 140)

## 1. Rohdaten einlesen

Die Datei ist tab-getrennt und enthält keinen Header. Sie wird explizit als UTF-8 gelesen, damit Zeichen wie `£`, `ü` oder typografische Apostrophe korrekt erhalten bleiben.

In [ ]:
df_raw = pd.read_csv(
    RAW_PATH,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8",
)

display(df_raw.head(10))
print(f"Original rows: {len(df_raw)}")

## 2. Labels normalisieren, HTML-Entities dekodieren und Target erzeugen

`html.unescape` dekodiert HTML-Entities wie `&lt;`, `&gt;` oder `&#gt;`. Das ist kein UTF-8-Mojibake, sondern HTML-Encoding innerhalb einzelner SMS-Texte.

In [ ]:
df = df_raw.copy()
df["label"] = df["label"].astype("string").str.strip().str.lower()
df["message"] = df["message"].astype("string").str.strip()
df["message"] = df["message"].map(lambda value: html.unescape(value) if pd.notna(value) else value)
df["target"] = df["label"].map({"ham": 0, "spam": 1}).astype("Int64")
df = df[["label", "target", "message"]]

display(df.head(10))
display(df.dtypes.to_frame("dtype"))

## 3. Anzahl und Klassenverteilung

In [ ]:
class_distribution = (
    df["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)
class_distribution["percentage"] = (class_distribution["count"] / len(df) * 100).round(2)
display(class_distribution)

print("Expected from dataset readme: 5,574 rows, 4,827 ham, 747 spam.")
print(f"Actual: {len(df)} rows")

Die Klassen sind deutlich unausgeglichen. SPAM macht nur etwa 13,4% der Originaldaten aus. Für spätere Modelle sind deshalb Precision, Recall und F1 für SPAM wichtiger als reine Accuracy.

## 4. Missing Values, leere Texte und ungültige Targets

In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing_values": df[["label", "target", "message"]].isna().sum(),
        "empty_or_whitespace": [
            df["label"].astype("string").str.strip().eq("").sum(),
            0,
            df["message"].astype("string").str.strip().eq("").sum(),
        ],
    },
    index=["label", "target", "message"],
)
display(missing_summary)

problem_rows = df[
    df["label"].isna()
    | df["target"].isna()
    | df["message"].isna()
    | df["label"].astype("string").str.strip().eq("")
    | df["message"].astype("string").str.strip().eq("")
]

print(f"Rows with missing/empty label, target or message: {len(problem_rows)}")
display(problem_rows.head(20))

## 5. Encoding-, ASCII- und HTML-Entity-Checks

Nicht-ASCII-Zeichen sind in SMS normal, z. B. `£` oder `ü`. Problematisch sind Mojibake-Marker wie `Â`, `Ã` oder `â`, wenn sie durch falsches Decoding entstanden sind. Zusätzlich prüfen wir, ob nach `html.unescape` noch HTML-Entities wie `&lt;` oder `&#gt;` übrig sind.

In [ ]:
mojibake_markers = ["Ã", "Â", "â€", "â€™", "â€œ", "â€�", "â€“", "â€”"]
mojibake_pattern = re.compile("|".join(re.escape(marker) for marker in mojibake_markers))
literal_escape_pattern = re.compile(r"\\x[0-9A-Fa-f]{2}|\\u[0-9A-Fa-f]{4}")
html_entity_pattern = re.compile(r"&(?:[a-zA-Z][a-zA-Z0-9]+|#[0-9]+|#x[0-9A-Fa-f]+);")
non_ascii_pattern = re.compile(r"[^\x00-\x7F]")

encoding_checks = df.assign(
    has_non_ascii=df["message"].str.contains(non_ascii_pattern, na=False),
    has_mojibake=df["message"].str.contains(mojibake_pattern, na=False),
    has_literal_escape=df["message"].str.contains(literal_escape_pattern, na=False),
    has_html_entity=df["message"].str.contains(html_entity_pattern, na=False),
)

encoding_summary = pd.DataFrame(
    {
        "check": [
            "messages with non-ASCII characters",
            "messages with likely mojibake markers",
            "messages with literal ASCII escape sequences",
            "messages with remaining HTML entities",
        ],
        "count": [
            int(encoding_checks["has_non_ascii"].sum()),
            int(encoding_checks["has_mojibake"].sum()),
            int(encoding_checks["has_literal_escape"].sum()),
            int(encoding_checks["has_html_entity"].sum()),
        ],
    }
)
display(encoding_summary)

display(encoding_checks.loc[encoding_checks["has_non_ascii"], ["label", "message"]].head(15))

mojibake_examples = encoding_checks.loc[encoding_checks["has_mojibake"], ["label", "message"]].head(20)
html_entity_examples = encoding_checks.loc[encoding_checks["has_html_entity"], ["label", "message"]].head(20)

print(f"Likely mojibake examples: {len(mojibake_examples)} shown")
display(mojibake_examples)
print(f"Remaining HTML entity examples: {len(html_entity_examples)} shown")
display(html_entity_examples)

## 6. Duplikate prüfen und entfernen

Duplikate werden nach dem bereinigten Nachrichtentext erkannt. Wenn ein identischer Text mehrfach vorkommt, behalten wir die erste Zeile und entfernen die weiteren Vorkommen.

In [ ]:
duplicate_rows = df[df.duplicated(subset="message", keep=False)].copy()

duplicate_summary = (
    duplicate_rows.groupby("message")
    .agg(
        occurrences=("message", "size"),
        labels=("label", lambda labels: ", ".join(sorted(labels.unique()))),
    )
    .reset_index()
    .sort_values(["occurrences", "message"], ascending=[False, True])
)

label_conflicts = duplicate_summary[duplicate_summary["labels"].str.contains(",", na=False)]
duplicate_extra_rows = int(duplicate_summary["occurrences"].sub(1).sum()) if len(duplicate_summary) else 0

print(f"Duplicate groups: {len(duplicate_summary)}")
print(f"Duplicate rows to remove: {duplicate_extra_rows}")
print(f"Duplicate texts with label conflicts: {len(label_conflicts)}")

display(duplicate_summary.head(20))
display(label_conflicts.head(20))

df_clean = df.drop_duplicates(subset="message", keep="first").reset_index(drop=True)
print(f"Rows before deduplication: {len(df)}")
print(f"Rows after deduplication: {len(df_clean)}")

## 7. Clean CSV schreiben

In [ ]:
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False, encoding="utf-8")
print(f"Saved: {CLEAN_PATH}")

df_check = pd.read_csv(CLEAN_PATH, encoding="utf-8")
display(df_check.head())
print(f"Rows written: {len(df_check)}")
print(f"Missing values after reload: {int(df_check[['label', 'target', 'message']].isna().sum().sum())}")

## 8. Original vs. Cleaned nebeneinander

Diese Übersicht zeigt, wie sich das Cleaning auf Zeilenanzahl, Klassenverteilung, Missing Values, HTML-Entities und Duplikate auswirkt.

In [ ]:
def dataset_profile(dataframe, name):
    return pd.Series(
        {
            "rows": len(dataframe),
            "ham": int((dataframe["label"] == "ham").sum()),
            "spam": int((dataframe["label"] == "spam").sum()),
            "missing_label": int(dataframe["label"].isna().sum()),
            "missing_target": int(dataframe["target"].isna().sum()),
            "missing_message": int(dataframe["message"].isna().sum()),
            "remaining_html_entities": int(dataframe["message"].astype("string").str.contains(html_entity_pattern, na=False).sum()),
            "likely_mojibake_markers": int(dataframe["message"].astype("string").str.contains(mojibake_pattern, na=False).sum()),
            "duplicate_extra_rows": int(dataframe.duplicated(subset="message").sum()),
        },
        name=name,
    )

comparison = pd.concat(
    [dataset_profile(df, "original_after_basic_cleaning"), dataset_profile(df_clean, "cleaned_written_to_csv")],
    axis=1,
)
display(comparison)

class_comparison = pd.concat(
    [
        df["label"].value_counts().rename("original_after_basic_cleaning"),
        df_clean["label"].value_counts().rename("cleaned_written_to_csv"),
    ],
    axis=1,
)
class_comparison["removed"] = class_comparison["original_after_basic_cleaning"] - class_comparison["cleaned_written_to_csv"]
display(class_comparison)